In [ ]:
!pip install datatable plotnine

In [ ]:
import sys
import pandas as pd
import datatable as dt
import numpy as np
from multiprocessing import Pool

PATH = "/home/jmurga/mkt/202004" 
sys.path.insert(0, PATH + '/scripts/src/')
from pyAmkt import *
from slimParser import *

Create the output directory. You should change the directory tree if you clone the github repo

In [ ]:
!mkdir -p /home/jmurga/mkt/202004/rawData/simulations /home/jmurga/mkt/202004/rawData/summStat

# No demography simulations

## N = 1000

### Solving values to simulate with Analytical.jl

In [ ]:
!which julia

Add the proper packages to your Julia session

In [ ]:
!julia -e 'using Pkg;Pkg.add("CSV");Pkg.add("DataFrames");Pkg.add(PackageSpec(path="https://github.com/jmurga/Analytical.jl"))'

In [ ]:
!julia /home/jmurga/mkt/202004/scripts/src/simTable.jl 1000 100000 bgsTable.tsv

In [ ]:
simTable = pd.read_csv(PATH + "/rawData/simulations/bgsTable.tsv", sep='\t')
simTable['path'] = simTable.apply(lambda row: "/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_" + str(row['alpha']) + "_" + str(row['alphaW']) + "_" + str(np.round(row['B'],4)),axis=1)

I test two cases in the notebook case I select one row from simTable

In [ ]:
tmpSim = simTable.tail(2);tmpSim

### Running SLiM

Generating replicates and folders. It depends on GNU parallel tool and SLiM

In [ ]:
binnedSfs=1000

In [ ]:
runSlim(recipe="/home/jmurga/mkt/202004/scripts/slimRecipes/bgs.slim",simTable=tmpSim.head(1),pSize=1000,codingLength=10000,strongStrength=500,weaklyStrength=10,bins=binnedSfs,replicas=[1,10000],threads=5,slimPath="/home/jmurga/.conda/envs/abcmk/bin/slim",parallelPath="/home/jmurga/.conda/envs/abcmk/bin/parallel")

In [ ]:
runSlim(recipe="/home/jmurga/mkt/202004/scripts/slimRecipes/bgs.slim",simTable=tmpSim.tail(1),pSize=1000,codingLength=10000,strongStrength=500,weaklyStrength=10,bins=binnedSfs,replicas=[1,10000],threads=5,slimPath="/home/jmurga/.conda/envs/abcmk/bin/slim",parallelPath="/home/jmurga/.conda/envs/abcmk/bin/parallel")

### Processing simulated data

In [ ]:
for index,row in tmpSim.iterrows():
    parsePolDiv(row.path,binnedSfs)

In [ ]:
out = [];al = [];
for index,row in tmpSim.iterrows():
    print(row.path)
    sfs = pd.read_csv(row.path + "/sfs.tsv",header=0,sep="\t")
    div = pd.read_csv(row.path + "/div.tsv",header=0,sep="\t")
    # alphas = pd.read_csv(row.path + "/alphas.tsv",header=0,sep="\t")
    alphas = div.dw/div.di
    try:
        cSfs = cumulativeSfs(sfs.to_numpy())
        asymp1 = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)
        asymp2 = amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
    except:
        
        cSfs = cumulativeSfs(reduceSfs(sfs.to_numpy(),100))
        asymp1 = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)
        try:            
            asymp2 = amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
        except:
            asymp2 = [None] * 2
            asymp2[0] = dict({'alpha':0})
            asymp2[1]= np.zeros(100)
            
    tmp1 = pd.DataFrame({'trueAlphaW':alphas})
    tmp1['asymp_nopos'] = asymp1[0]['alpha']
    tmp1['asymp'] = asymp2[0]['alpha']
    tmp1['analyticalEstimation']= row.estimation
    tmp1['path'] = row.path.split('/')[-1]
    tmp2 = pd.DataFrame({'path':row.path.split('/')[-1],'alpha_nopos':asymp1[1],'alpha':asymp2[1],'f':np.arange(1,101)}) 
    out.append(tmp1)
    al.append(tmp2)
df = pd.concat(out).reset_index(drop=True)
alphas = pd.concat(al)


In [ ]:
df

### Julia analysis

In [ ]:
iteration=round(10**2/17/4)+1

Each Julia iteration performs 17 estimations accounting for 17 different BGS values regarding a fixed alpha value. To make about 10^6 summary statistics we parallelize (10^6/17/threads) iterations. Each thread will write a different file that I will join using bash to input in ABCreg. I will storaged a file called rJob.sh which should be executed to perform the ABC.

In [ ]:
priorsJulia(table=tmpSim,nSimulations=iteration,model="noDemog",script="/home/jmurga/mkt/202004/scripts/src/sim.jl",threads=4,replicas=[1,4],precomipledImg="/home/jmurga/mkt/202004/scripts/src/mktest.so",parallelPath="/home/jmurga/.conda/envs/abcmk/bin/parallel")

# Tennesen model

## Solving values to simulate with Analytical.jl

In [ ]:
!which julia

In [ ]:
!julia /home/jmurga/mkt/202004/scripts/src/simTable.jl 7310 100000 tennensen.tsv

### Solving values to simulate with Analytical.jl

In [ ]:
simTable = pd.read_csv(PATH + "/rawData/simulations/tennesen.tsv", sep='\t')
simTable['path'] = simTable.apply(lambda row: "/home/jmurga/mkt/202004/rawData/simulations/tennesen/tennesen_" + str(row['alpha']) + "_" + str(row['alphaW']) + "_" + str(np.round(row['B'],4)),axis=1)

To test just one case I select one row from simTable

In [ ]:
tmpSim = simTable.tail(2);tmpSim

### Running SLiM

Generating replicates and folders. It depends on GNU parallel tool and SLiM

In [ ]:
binnedSfs=1000

In [ ]:
for index,row in tmpSim.iterrows():
    print(index,row.path);tmp = pd.DataFrame(row).T
    
    runSlim(recipe="/home/jmurga/mkt/202004/scripts/slimRecipes/tennesen.slim",simTable=tmp,pSize=1000,codingLength=10000,strongStrength=500,weaklyStrength=10,bins=binnedSfs,replicas=[1,100000],threads=5,slimPath="/home/jmurga/.conda/envs/abcmk/bin/slim",parallelPath="/home/jmurga/.conda/envs/abcmk/bin/parallel")

### Processing simulated data

In [ ]:
for index,row in tmpSim.iterrows():
    parsePolDiv(row.path,binnedSfs)

In [ ]:
out = [];al = [];
for index,row in tmpSim.iterrows():
    print(row.path)
    sfs = pd.read_csv(row.path + "/sfs.tsv",header=0,sep="\t")
    div = pd.read_csv(row.path + "/div.tsv",header=0,sep="\t")
    # alphas = pd.read_csv(row.path + "/alphas.tsv",header=0,sep="\t")
    alphas = div.dw/div.di
    try:
        cSfs = cumulativeSfs(sfs.to_numpy())
        asymp1 = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)
        asymp2 = amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
    except:
        
        cSfs = cumulativeSfs(reduceSfs(sfs.to_numpy(),100))
        asymp1 = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)
        try:            
            asymp2 = amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
        except:
            asymp2 = [None] * 2
            asymp2[0] = dict({'alpha':0})
            asymp2[1]= np.zeros(100)
            
    tmp1 = pd.DataFrame({'trueAlphaW':alphas})
    tmp1['asymp_nopos'] = asymp1[0]['alpha']
    tmp1['asymp'] = asymp2[0]['alpha']
    tmp1['analyticalEstimation']= row.estimation
    tmp1['path'] = row.path.split('/')[-1]
    tmp2 = pd.DataFrame({'path':row.path.split('/')[-1],'alpha_nopos':asymp1[1],'alpha':asymp2[1],'f':np.arange(1,101)}) 
    out.append(tmp1)
    al.append(tmp2)
df = pd.concat(out).reset_index(drop=True)
alphas = pd.concat(al)

In [ ]:
df

### Julia Analysis

In [ ]:
iteration=(10**6/17/4)

Each Julia iteration performs 17 estimations accounting for 17 different BGS values regarding a fixed alpha value. To make about 10^6 summary statistics we parallelize (10^6/17/threads) iterations. Each thread will write a different file that I will join using bash to input in ABCreg. I will storaged a file called rJob.sh which should be executed to perform the ABC.

In [ ]:
priorsJulia(table=tmpSim,nSimulations=iteration,model="tennesen",script="/home/jmurga/mkt/202004/scripts/src/sim.jl",threads=4,replicas=[1,4],precomipledImg="/home/jmurga/mkt/202004/scripts/src/mktest.so",parallelPath="/home/jmurga/.conda/envs/abcmk/bin/parallel")